# AutoShot keyframe embedding baseline

Pipeline đơn giản: đọc dataset do `get-keyframe-autoshot.ipynb` tạo → embedding keyframe bằng OpenCLIP → lưu `.npy`, mapping CSV và `model_info.json`.

In [1]:
!pip install -q open_clip_torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0

In [2]:
from pathlib import Path
from zipfile import ZipFile
from datetime import datetime, timezone
import json

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch
import open_clip

AUTOSHOT_ROOT = Path('/kaggle/input/datasets/khngxuninh/autoshot-output')
OUTPUT_DIR = Path('/kaggle/working/embedding')

SELECTED_STRATEGY = 'fast_baseline'
STRATEGY_CONFIGS = {
    'fast_baseline': {
        'model': 'ViT-B-32',
        'pretrained': 'laion2b_s34b_b79k',
        'batch_size': 256,
    },
    'classic_quality': {
        'model': 'ViT-L-14',
        'pretrained': 'laion2b_s32b_b82k',
        'batch_size': 128,
    },
    'pe_experiment': {
        'model': 'hf-hub:timm/PE-Core-B-16',
        'pretrained': None,
        'batch_size': 128,
    },
}

selected_config = STRATEGY_CONFIGS[SELECTED_STRATEGY]
MODEL_NAME = selected_config['model']
PRETRAINED = selected_config['pretrained']
BATCH_SIZE = selected_config['batch_size']
NUM_WORKERS = 4
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Device:', DEVICE)
print('AutoShot root:', AUTOSHOT_ROOT)


Device: cuda
AutoShot root: /kaggle/input/datasets/khngxuninh/autoshot-output


## 1. Đọc metadata và tạo mapping

Mỗi dòng `saved=True` trong `shot_segments.csv` tương ứng một ảnh first/middle/last của một shot.

In [3]:
metadata_path = AUTOSHOT_ROOT / 'shot_segments.csv'
frames_root = AUTOSHOT_ROOT / 'frames'

mapping = pd.read_csv(metadata_path)
mapping = mapping[mapping['saved'].astype(str).str.lower() == 'true'].reset_index(drop=True)

# image_path trong CSV là path của notebook cũ; dựng lại path trong dataset hiện tại.
relative_paths = [
    (Path('frames') / Path(old_path).parent.name / Path(old_path).name).as_posix()
    for old_path in mapping['image_path'].astype(str)
]
image_paths = [AUTOSHOT_ROOT / path for path in relative_paths]

mapping.insert(0, 'embedding_index', np.arange(len(mapping)))
mapping.insert(1, 'video_id', mapping['video_name'].map(lambda name: Path(name).stem))
mapping['relative_path'] = relative_paths
mapping = mapping.drop(columns=['image_path', 'saved'])

print(f'Số keyframe: {len(mapping):,}')
display(mapping.head())


Số keyframe: 13,278


,embedding_index,video_id,video_name,video_path,shot_id,shot_start_frame,shot_end_frame,shot_start_sec,shot_end_sec,frame_type,frame_idx,frame_sec,boundary_threshold,fps,total_frames_opencv,relative_path
0,0,L30_V071,L30_V071.mp4,/kaggle/input/datasets/aresusayhi/ai-challenge...,0,0,74,0.0,2.96,first,0,0.00,0.296,25.0,1942,frames/L30_V071/shot_0000_first_f000000.jpg
1,1,L30_V071,L30_V071.mp4,/kaggle/input/datasets/aresusayhi/ai-challenge...,0,0,74,0.0,2.96,middle,37,1.48,0.296,25.0,1942,frames/L30_V071/shot_0000_middle_f000037.jpg
2,2,L30_V071,L30_V071.mp4,/kaggle/input/datasets/aresusayhi/ai-challenge...,0,0,74,0.0,2.96,last,74,2.96,0.296,25.0,1942,frames/L30_V071/shot_0000_last_f000074.jpg
3,3,L30_V071,L30_V071.mp4,/kaggle/input/datasets/aresusayhi/ai-challenge...,1,75,164,3.0,6.56,first,75,3.00,0.296,25.0,1942,frames/L30_V071/shot_0001_first_f000075.jpg
4,4,L30_V071,L30_V071.mp4,/kaggle/input/datasets/aresusayhi/ai-challenge...,1,75,164,3.0,6.56,middle,119,4.76,0.296,25.0,1942,frames/L30_V071/shot_0001_middle_f000119.jpg


## 2. Load OpenCLIP và DataLoader

In [4]:
class KeyframeDataset(torch.utils.data.Dataset):
    def __init__(self, paths, transform):
        self.paths = paths
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            return self.transform(image.convert('RGB'))

if PRETRAINED is None:
    model, _, preprocess = open_clip.create_model_and_transforms(
        MODEL_NAME, device=DEVICE
    )
else:
    model, _, preprocess = open_clip.create_model_and_transforms(
        MODEL_NAME, pretrained=PRETRAINED, device=DEVICE
    )
model.eval()

dataset = KeyframeDataset(image_paths, preprocess)
loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == 'cuda'),
)

print(f'{MODEL_NAME} / {PRETRAINED} | {len(loader):,} batches')


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

ViT-B-32 / laion2b_s34b_b79k | 52 batches


## 3. Tạo embedding

In [5]:
embedding_batches = []

with torch.inference_mode():
    for images in tqdm(loader, desc='Embedding'):
        images = images.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE, enabled=(DEVICE == 'cuda')):
            features = model.encode_image(images, normalize=True)
        embedding_batches.append(features.float().cpu().numpy())

embeddings = np.concatenate(embedding_batches).astype('float32')
print('Embedding shape:', embeddings.shape)


Embedding:   0%|          | 0/52 [00:00<?, ?it/s]

Embedding shape: (13278, 512)


## 4. Lưu output

In [6]:
embedding_path = OUTPUT_DIR / f'keyframe_embeddings_{SELECTED_STRATEGY}.npy'
mapping_path = OUTPUT_DIR / 'keyframe_mapping.csv'
info_path = OUTPUT_DIR / f'model_info_{SELECTED_STRATEGY}.json'

np.save(embedding_path, embeddings)
mapping.to_csv(mapping_path, index=False)

model_info = {
    'dataset': 'khngxuninh/autoshot-output',
    'keyframe_extractor': 'AutoShot',
    'keyframe_policy': 'first, middle, last frame of each shot',
    'strategy': SELECTED_STRATEGY,
    'model_name': MODEL_NAME,
    'pretrained': PRETRAINED,
    'embedding_shape': list(embeddings.shape),
    'embedding_dtype': str(embeddings.dtype),
    'l2_normalized': True,
    'batch_size': BATCH_SIZE,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
}

with info_path.open('w', encoding='utf-8') as f:
    json.dump(model_info, f, ensure_ascii=False, indent=2)

print('Saved:')
print('-', embedding_path)
print('-', mapping_path)
print('-', info_path)


Saved:
- /kaggle/working/embedding/keyframe_embeddings_fast_baseline.npy
- /kaggle/working/embedding/keyframe_mapping.csv
- /kaggle/working/embedding/model_info_fast_baseline.json


# So sánh các chiến lược embedding

Benchmark cùng một tập keyframe mẫu để so sánh chi phí chạy. Các chỉ số gồm thời gian load, thời gian inference, ảnh/giây, peak VRAM, số tham số, dimension và dung lượng `.npy` ước tính khi chạy toàn bộ dataset.

- **Fast baseline:** ViT-B/32 — nhẹ và nhanh.
- **Classic quality:** ViT-L/14 — backbone lớn hơn, thường cho retrieval tốt hơn nhưng chậm hơn.
- **PE experiment:** PE-Core-B/16 — Perception Encoder mới hơn, vector 1024 chiều.

> Benchmark này chỉ so sánh tài nguyên và tốc độ. Muốn kết luận model nào tìm kiếm chính xác hơn cần một tập text query cùng ground truth để tính Recall@K/mAP. Thời gian load lần đầu có thể bao gồm thời gian tải checkpoint.

In [7]:
import gc
import time

BENCHMARK_SIZE = min(2048, len(image_paths))
benchmark_paths = image_paths[:BENCHMARK_SIZE]

STRATEGIES = [
    {'strategy': name, **config}
    for name, config in STRATEGY_CONFIGS.items()
]

print(f'Benchmark {BENCHMARK_SIZE:,}/{len(image_paths):,} keyframes')


Benchmark 2,048/13,278 keyframes


In [8]:
# Giải phóng baseline model trước khi đo VRAM từng chiến lược.
for variable_name in ['model', 'loader', 'dataset']:
    globals().pop(variable_name, None)
gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

benchmark_rows = []

for config in STRATEGIES:
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    load_start = time.perf_counter()
    if config['pretrained'] is None:
        bench_model, _, bench_preprocess = open_clip.create_model_and_transforms(
            config['model'], device=DEVICE
        )
    else:
        bench_model, _, bench_preprocess = open_clip.create_model_and_transforms(
            config['model'], pretrained=config['pretrained'], device=DEVICE
        )
    bench_model.eval()
    load_seconds = time.perf_counter() - load_start

    bench_dataset = KeyframeDataset(benchmark_paths, bench_preprocess)
    bench_loader = torch.utils.data.DataLoader(
        bench_dataset,
        batch_size=config['batch_size'],
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == 'cuda'),
    )

    processed = 0
    dimension = None
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    inference_start = time.perf_counter()

    with torch.inference_mode():
        for images in tqdm(bench_loader, desc=config['strategy']):
            images = images.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE, enabled=(DEVICE == 'cuda')):
                features = bench_model.encode_image(images, normalize=True)
            processed += len(images)
            dimension = features.shape[1]

    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    inference_seconds = time.perf_counter() - inference_start
    peak_vram_gb = (
        torch.cuda.max_memory_allocated() / 1024**3 if DEVICE == 'cuda' else 0
    )
    parameter_millions = sum(p.numel() for p in bench_model.parameters()) / 1e6

    benchmark_rows.append({
        'strategy': config['strategy'],
        'model': config['model'],
        'pretrained': config['pretrained'] or 'hf-hub',
        'batch_size': config['batch_size'],
        'sample_images': processed,
        'dimension': dimension,
        'parameters_million': round(parameter_millions, 1),
        'load_seconds': round(load_seconds, 2),
        'inference_seconds': round(inference_seconds, 2),
        'images_per_second': round(processed / inference_seconds, 2),
        'peak_vram_gb': round(peak_vram_gb, 2),
        'estimated_full_hours': round(
            inference_seconds / processed * len(image_paths) / 3600, 2
        ),
        'estimated_float32_npy_gb': round(
            len(image_paths) * dimension * 4 / 1024**3, 3
        ),
    })

    del bench_model, bench_preprocess, bench_loader, bench_dataset, features

benchmark_results = pd.DataFrame(benchmark_rows)
benchmark_results.to_csv(OUTPUT_DIR / 'embedding_benchmark.csv', index=False)
display(benchmark_results)


fast_baseline:   0%|          | 0/8 [00:00<?, ?it/s]

open_clip_pytorch_model.bin:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

classic_quality:   0%|          | 0/16 [00:00<?, ?it/s]

open_clip_config.json:   0%|          | 0.00/600 [00:00<?, ?B/s]

open_clip_model.safetensors:   0%|          | 0.00/1.79G [00:00<?, ?B/s]

pe_experiment:   0%|          | 0/16 [00:00<?, ?it/s]

,strategy,model,pretrained,batch_size,sample_images,dimension,parameters_million,load_seconds,inference_seconds,images_per_second,peak_vram_gb,estimated_full_hours,estimated_float32_npy_gb
0,fast_baseline,ViT-B-32,laion2b_s34b_b79k,256,2048,512,151.3,1.97,33.34,61.43,1.11,0.06,0.025
1,classic_quality,ViT-L-14,laion2b_s32b_b82k,128,2048,768,427.6,14.60,37.11,55.19,2.99,0.07,0.038
2,pe_experiment,hf-hub:timm/PE-Core-B-16,hf-hub,128,2048,1024,447.7,23.61,19.04,107.54,2.42,0.03,0.051
